# Semana 8 · PyTorch

**Nivel Intermedio** · Semana 8 · Clase expositiva [A] · 2 horas · CPU; la parte 7 compara con una GPU T4 (opcional)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/clases/sem08_pytorch.ipynb)

**Objetivos.** Al terminar la clase podrás:

1. Pasar de NumPy a PyTorch: crear tensores, elegir su tipo (`float32`, `int64`) y traducir `axis` a `dim`.
2. Usar `autograd` para obtener gradientes con `loss.backward()`, y reconocer en él la vuelta que hiciste a mano en la semana 7.
3. Armar una red con `nn.Module` y entrenarla con `DataLoader`, un optimizador y el bucle de entrenamiento de cinco líneas.
4. Mover un modelo y sus datos a la GPU, y explicar cuándo conviene y cuándo no.

**Prerrequisitos.** Semana 1: clases, `__init__`, `__call__` y la clase `Linear` con `forward`. Semana 2: formas, *broadcasting*, `axis`. Semana 4: descenso de gradiente y entropía cruzada. Semanas 6 y 7: la red `64 → 32 → 10` de los dígitos, su ida y su vuelta en NumPy.

**Hoy cambia la herramienta, no las ideas.** Todo lo que vamos a usar ya lo construiste a mano: la capa (semana 1), la ida (semana 6), la vuelta (semana 7 y P1). PyTorch lo hace más rápido, para cualquier red, y también en la GPU.

**Agenda** (después de la apertura de 15 minutos, donde se entrega P1).

| Parte | Tema | Minutos |
|---|---|---|
| 1 | Tensores: arreglos de NumPy con dos superpoderes | 15 |
| 2 | `autograd`: la vuelta de la semana 7, automática | 20 |
| 3 | `nn.Module`: la red como piezas | 15 |
| 4 | Pérdida, optimizador y el bucle de cinco líneas | 15 |
| 5 | `Dataset` y `DataLoader`: aprender de a bocados | 15 |
| 6 | Un problema de verdad: MNIST | 10 |
| 7 | CPU vs GPU | 10 |
| 8 | Cierre y tareas para la semana 9 | 5 |

Ejecuta esta celda primero: carga las librerías y los dígitos de las semanas 6 y 7, fija las semillas y define `check`, la función que revisa tus ejercicios de "Tu turno".

En Colab, PyTorch ya viene instalado: no hace falta `pip install`.

In [ ]:
import copy
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import datasets

SEED = 0
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data / 16, digits.target, test_size=0.25, stratify=digits.target, random_state=SEED)

PENGUINS_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"


def check(name, test):
    """Ejecuta una prueba y muestra ✓, ✗ o "todavía sin resolver", sin detener el notebook."""
    try:
        test()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"… {name}: todavía sin resolver")
    except AssertionError as e:
        print(f"✗ {name}: {e}")
    except Exception as e:
        print(f"✗ {name}: {type(e).__name__}: {e}")


print("PyTorch", torch.__version__, "| ¿hay GPU?", torch.cuda.is_available())
print("dígitos:", X_train.shape, "para entrenar,", X_test.shape, "para probar")

## 1. Tensores: arreglos de NumPy con dos superpoderes

Un **tensor** de PyTorch es, para empezar, lo mismo que un arreglo de NumPy: una caja de números con una forma (*shape*). Todo lo de la semana 2 sigue valiendo: indexar, `reshape`, *broadcasting*, `@`. La diferencia son dos superpoderes que NumPy no tiene:

1. **Memoria.** Si se lo pides, un tensor recuerda con qué operaciones se calculó. Con esa memoria, PyTorch recorre la cuenta hacia atrás y saca los gradientes solo (parte 2).
2. **Pasaporte.** Un tensor puede viajar a la GPU y hacer allá sus cuentas, muchísimo más rápido cuando son grandes (parte 7).

La traducción es casi palabra por palabra:

| NumPy | PyTorch |
|---|---|
| `np.array([[1.0, 2.0]])` | `torch.tensor([[1.0, 2.0]])` |
| `a.shape`, `a.reshape(...)`, `a.T`, `a @ b` | igual |
| `a.sum(axis=0)` | `t.sum(dim=0)` |
| `keepdims=True` | `keepdim=True` |
| `np.zeros((2, 3))`, `rng.normal(size=(2, 3))` | `torch.zeros(2, 3)`, `torch.randn(2, 3)` |
| `np.concatenate`, `np.exp`, `np.maximum(z, 0)` | `torch.cat`, `torch.exp`, `torch.relu(z)` |

In [ ]:
a = np.array([[1.0, 2.0], [3.0, 4.0]])
t = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("NumPy:  ", a.shape, a.dtype)
print("PyTorch:", t.shape, t.dtype)

print("\nt @ t =\n", t @ t)
print("t.sum(dim=0) =", t.sum(dim=0))                  # en NumPy: a.sum(axis=0)
print("t + torch.tensor([10.0, 20.0]) =\n", t + torch.tensor([10.0, 20.0]))   # broadcasting, como en la semana 2

Mira los tipos: NumPy usa `float64` y PyTorch, `float32`. No es un detalle. Un `float32` ocupa la mitad de memoria, las GPU son mucho más rápidas con él, y sus 7 cifras significativas alcanzan de sobra para entrenar redes. Por eso todo en PyTorch es `float32` por defecto.

El problema aparece al cruzar el puente. `torch.from_numpy` conserva el tipo de NumPy, y PyTorch no mezcla tipos en una multiplicación: es el primer error que vas a ver con datos reales.

In [ ]:
X_wrong = torch.from_numpy(X_train[:5])     # conserva float64
W = torch.randn(64, 10)                      # float32
print("X:", X_wrong.dtype, "| W:", W.dtype)
try:
    X_wrong @ W
except RuntimeError as e:
    print("Error:", e)

X_ok = torch.tensor(X_train[:5], dtype=torch.float32)
print("Con dtype=torch.float32:", tuple((X_ok @ W).shape))

La regla práctica: **las entradas, en `float32`; las etiquetas de clase, en `int64`** (enteros), que es lo que pide la entropía cruzada de PyTorch. Convertimos los dígitos una vez y los usamos el resto de la clase.

Para volver: `.numpy()` da un arreglo de NumPy, y `.item()` saca un número suelto de un tensor de un solo elemento.

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train)              # enteros de NumPy → int64
y_test_t = torch.tensor(y_test)

print("X_train_t:", tuple(X_train_t.shape), X_train_t.dtype)
print("y_train_t:", tuple(y_train_t.shape), y_train_t.dtype)
print("de vuelta a NumPy:", type(X_train_t.numpy()).__name__, "| un número suelto:", y_train_t[0].item())

### Tu turno 1: *softmax* en PyTorch

En la semana 4 escribiste *softmax* en NumPy: para cada fila, restar su máximo (para que `exp` no explote), tomar `exp` y dividir por la suma de la fila. Tradúcela a PyTorch.

Completa `softmax_rows(Z)`: recibe un tensor de forma `(n, k)` y devuelve otro de la misma forma y el mismo tipo, donde cada fila suma 1. Usa `dim=1` y `keepdim=True`. Ojo: en PyTorch, `Z.max(dim=1)` devuelve **dos** cosas, los valores y sus posiciones; los valores son `Z.max(dim=1, keepdim=True).values`.

In [ ]:
def softmax_rows(Z):
    # TODO
    raise NotImplementedError

In [ ]:
def test_softmax():
    Z = torch.randn(6, 4, generator=torch.Generator().manual_seed(1))
    P = softmax_rows(Z)
    assert isinstance(P, torch.Tensor), "debe devolver un tensor de PyTorch, no un arreglo de NumPy"
    assert P.shape == Z.shape, f"la forma debería ser {tuple(Z.shape)}, es {tuple(P.shape)}"
    assert P.dtype == torch.float32, f"debería seguir en float32, está en {P.dtype}"
    assert torch.allclose(P.sum(dim=1), torch.ones(6)), "cada fila debe sumar 1: ¿dividiste por la suma de la fila?"
    assert torch.allclose(P, torch.softmax(Z, dim=1), atol=1e-6), "no coincide con torch.softmax"


def test_softmax_big():
    P = softmax_rows(torch.tensor([[1000.0, 1001.0], [-1000.0, -1000.0]]))
    assert not torch.isnan(P).any(), "con números grandes da nan: resta el máximo de cada fila antes de exp"
    assert torch.allclose(P, torch.tensor([[0.2689, 0.7311], [0.5, 0.5]]), atol=1e-4), "con números grandes no da lo esperado"


check("softmax_rows", test_softmax)
check("softmax_rows con números grandes", test_softmax_big)

## 2. `autograd`: la vuelta de la semana 7, automática

Volvamos a la compra de la semana 7: 4 camisetas a 3 dólares, con 2 dólares de envío por camiseta. Ese día recorrimos el grafo a mano, de derecha a izquierda, y en P1 escribiste un motor que lo hace solo. PyTorch trae ese motor incluido, se llama **`autograd`** (diferenciación automática, *automatic differentiation*), y se usa en tres pasos:

1. Marca con `requires_grad=True` los números respecto de los cuales quieres derivar. Desde ahí, PyTorch anota cada operación en que participan: va dibujando el grafo mientras haces la cuenta.
2. Haz la cuenta normalmente: esa es la ida.
3. Llama a `.backward()` sobre el resultado. PyTorch recorre el grafo hacia atrás con la regla de siempre, gradiente que llega × derivada local, y deja cada gradiente en el atributo `.grad` de cada entrada.

In [ ]:
price = torch.tensor(3.0, requires_grad=True)
shipping = torch.tensor(2.0, requires_grad=True)
quantity = torch.tensor(4.0, requires_grad=True)

total = (price + shipping) * quantity     # la ida: PyTorch anota cada operación
total.backward()                          # la vuelta, por todo el grafo

print("total =", total.item())
print("d_price =", price.grad.item(), "| d_shipping =", shipping.grad.item(), "| d_quantity =", quantity.grad.item())

Los números de la pizarra: 4, 4 y 5. El grafo existe de verdad; cada resultado guarda en `grad_fn` el nodo que lo produjo, y cada nodo sabe de quién recibió sus entradas. `AccumulateGrad` es una hoja: un número que creaste tú, donde se acumula su `.grad`.

In [ ]:
node = total.grad_fn
print("total sale de:", type(node).__name__)
for parent, _ in node.next_functions:
    print("   que recibe de:", type(parent).__name__)
    for grandparent, _ in parent.next_functions:
        print("      que recibe de:", type(grandparent).__name__)

### La red de papel de la semana 7

La vuelta por la red $2 \to 2 \to 1$ nos tomó una tabla entera. Con `autograd` es escribir la ida y llamar a `backward`. Compara con la tabla: $\partial L/\partial W_1 = \begin{pmatrix} -0.730 & 0.365 \\ 0 & 0 \end{pmatrix}$, $\partial L/\partial W_2 = (-0.182,\ 0)$.

In [ ]:
x = torch.tensor([2.0, -1.0])
y = torch.tensor(1.0)
W1 = torch.tensor([[1.0, 2.0], [-1.0, 1.0]], requires_grad=True)
b1 = torch.tensor([1.0, 0.0], requires_grad=True)
W2 = torch.tensor([[2.0, -1.0]], requires_grad=True)
b2 = torch.tensor([-0.5], requires_grad=True)

p = torch.sigmoid(W2 @ torch.relu(W1 @ x + b1) + b2)[0]
loss = -(y * torch.log(p) + (1 - y) * torch.log(1 - p))
loss.backward()

for name, param in [("W1", W1), ("b1", b1), ("W2", W2), ("b2", b2)]:
    print(f"dL/d{name} = {(np.round(param.grad.double().numpy(), 3) + 0.0).tolist()}")

### Dos detalles que se aprenden a golpes

**Los gradientes se acumulan.** `backward` no reemplaza el `.grad` de antes: le **suma** el nuevo. Es la regla de las ramas de la semana 7 (si un valor se usa dos veces, sus gradientes se suman), y es útil, pero significa que entre un paso de entrenamiento y el siguiente hay que borrar los gradientes viejos. En P1 lo viste con `zero_grad`. Mira qué pasa si llamas a `backward` tres veces sin borrar:

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    loss = (w - 3) ** 2
    loss.backward()
    print(f"después del backward {i + 1}: w.grad = {w.grad.item()}")

w.grad.zero_()
print("después de w.grad.zero_():", w.grad.item())

La derivada de $(w - 3)^2$ en $w = 1$ es $-4$. La primera vez está bien; después, $-8$ y $-12$ son basura.

**El paso no es parte de la cuenta.** Al actualizar $w \leftarrow w - \eta\, \partial L/\partial w$, no queremos que PyTorch anote esa resta en el grafo. Para eso está `with torch.no_grad():`, un bloque donde PyTorch deja de anotar.

Con esos dos detalles, aquí está el descenso de gradiente de la semana 4, $L(w) = (w - 3)^2$ desde $w = 0$ con $\eta = 0.1$, **sin escribir la derivada**:

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
lr = 0.1
for step in range(1, 31):
    loss = (w - 3) ** 2           # 1. ida
    loss.backward()               # 2. vuelta: llena w.grad
    with torch.no_grad():
        w -= lr * w.grad          # 3. un paso cuesta abajo, sin anotarlo en el grafo
    w.grad.zero_()                # 4. borrar el gradiente para el próximo paso
    if step <= 4 or step % 10 == 0:
        print(f"paso {step:2d}: w = {w.item():.4f}")

Los mismos números de la semana 4: 0.6, 1.08, 1.464… hacia 3.

### Tu turno 2: la recta de los pingüinos, con `autograd`

En la semana 4 ajustaste una recta que predice la masa de un pingüino (en kg) a partir del largo de su aleta (estandarizado), y para eso derivaste a mano el MSE respecto de $w$ y de $b$. Hazlo de nuevo, pero sin derivar nada.

1. Crea `w` y `b` como tensores con `requires_grad=True`, empezando en $w = -0.5$, $b = 2.5$, como en la semana 4.
2. Da 60 pasos con $\eta = 0.1$. En cada paso: la pérdida `((w * x_peng + b - y_peng) ** 2).mean()`, `backward`, la actualización de **los dos** dentro de `torch.no_grad()`, y borrar **los dos** gradientes.
3. Guarda la pérdida de cada paso (con `.item()`) en la lista `peng_losses`.

Al final, `w` y `b` deben quedar donde quedaron en la semana 4.

In [ ]:
penguins = pd.read_csv(PENGUINS_URL).dropna(subset=["flipper_length_mm", "body_mass_g"])
flipper = penguins["flipper_length_mm"].to_numpy()
x_peng = torch.tensor((flipper - flipper.mean()) / flipper.std(), dtype=torch.float32)
y_peng = torch.tensor(penguins["body_mass_g"].to_numpy() / 1000, dtype=torch.float32)
print("pingüinos:", len(x_peng))

In [ ]:
peng_losses = []
# TODO: w, b y los 60 pasos

In [ ]:
def test_penguins():
    if not peng_losses:
        raise NotImplementedError
    xs, ys = x_peng.double().numpy(), y_peng.double().numpy()
    w_ref, b_ref = -0.5, 2.5
    for _ in range(60):
        err = w_ref * xs + b_ref - ys
        w_ref, b_ref = w_ref - 0.1 * 2 * np.mean(err * xs), b_ref - 0.1 * 2 * np.mean(err)
    assert len(peng_losses) == 60, f"peng_losses debería tener 60 pérdidas, tiene {len(peng_losses)}"
    assert peng_losses[-1] < peng_losses[0], "la pérdida no bajó"
    assert isinstance(w, torch.Tensor) and isinstance(b, torch.Tensor), "w y b deben ser tensores"
    assert abs(w.item() - w_ref) < 1e-3 and abs(b.item() - b_ref) < 1e-3, \
        f"w = {w.item():.3f}, b = {b.item():.3f}; deberían ser {w_ref:.3f} y {b_ref:.3f}. ¿Borras los dos gradientes en cada paso?"


check("la recta de los pingüinos", test_penguins)

## 3. `nn.Module`: la red como piezas

En la semana 1 escribiste una clase `Linear` con `weight`, `bias`, `forward` y `__call__`, y te dije que era el mismo patrón de PyTorch. Aquí está el original: `nn.Linear(entradas, salidas)`. Guarda `weight` con forma `(salidas, entradas)`, igual que nuestra `W` de la semana 6, y calcula $XW^\top + b$.

Las piezas de una red son como bloques de Lego: cada una sabe hacer su ida, y PyTorch sabe hacer la vuelta de todas.

In [ ]:
torch.manual_seed(SEED)
layer = nn.Linear(64, 32)
print(layer)
print("weight:", tuple(layer.weight.shape), "| bias:", tuple(layer.bias.shape))
print("¿layer(X) es X @ W.T + b?", torch.allclose(layer(X_train_t), X_train_t @ layer.weight.T + layer.bias))
print("¿sus pesos llevan gradiente?", layer.weight.requires_grad)

Dos cosas que `nn.Linear` hace por ti: sus pesos ya vienen con `requires_grad=True`, y ya vienen inicializados al azar con una escala razonable (cuál y por qué es tema de la semana 9).

Para apilar piezas en fila está `nn.Sequential`. Esta es la red de los dígitos de las semanas 6 y 7, `64 → 32 → 10`:

In [ ]:
torch.manual_seed(SEED)
model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10),
)
print(model)
print("parámetros:", sum(p.numel() for p in model.parameters()))
print("salida para 5 dígitos:", tuple(model(X_train_t[:5]).shape))

Los mismos 2410 parámetros de la semana 7, sin escribir una sola fórmula de la vuelta.

Cuando la red no es una fila simple, se escribe una clase que hereda de `nn.Module`, como en la semana 1:

- En `__init__`, primero `super().__init__()`, y después cada pieza como atributo. Al guardarla en `self`, PyTorch la **registra**: sus parámetros aparecen en `parameters()`.
- En `forward`, la ida. Nunca se llama a `forward` directamente: se llama al modelo, `net(x)`, y `__call__` llama a `forward` (semana 1).

In [ ]:
class TwoLayerNet(nn.Module):
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()                          # prepara la maquinaria de nn.Module
        self.hidden = nn.Linear(n_in, n_hidden)     # guardada como atributo: queda registrada
        self.output = nn.Linear(n_hidden, n_out)

    def forward(self, x):
        h = torch.relu(self.hidden(x))
        return self.output(h)


torch.manual_seed(SEED)
net = TwoLayerNet(64, 32, 10)
print(net)
for name, p in net.named_parameters():
    print(f"  {name:<14} {tuple(p.shape)}")

### Tu turno 3: un MLP de cualquier profundidad

Completa la clase `MLP(sizes)`. `sizes` es una lista como `[64, 32, 16, 10]`: entradas, capas ocultas y salidas. La red tiene una `nn.Linear` entre cada par de tamaños consecutivos y una ReLU después de cada capa **menos la última** (la última da los puntajes, como en la semana 6).

Puedes guardar las capas en un `nn.Sequential` (se le pueden pasar piezas con `*lista`) o en un `nn.ModuleList` y recorrerlas en `forward`. Lo que **no** sirve es una lista común de Python: PyTorch no mira dentro de ella, y sus parámetros no aparecen en `parameters()`. Pruébalo: el optimizador nunca los movería.

In [ ]:
class MLP(nn.Module):
    def __init__(self, sizes):
        super().__init__()
        # TODO

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
def run_by_hand(model, x):
    """La ida, capa por capa, con las nn.Linear que el modelo registró: ReLU entre capas, no después de la última."""
    linears = [m for m in model.modules() if isinstance(m, nn.Linear)]
    for i, lin in enumerate(linears):
        x = lin(x)
        if i < len(linears) - 1:
            x = torch.relu(x)
    return x


def test_mlp_structure():
    torch.manual_seed(1)
    m = MLP([64, 32, 16, 10])
    m(torch.randn(2, 64))
    linears = [mod for mod in m.modules() if isinstance(mod, nn.Linear)]
    assert len(linears) > 0, "PyTorch no ve ninguna nn.Linear: ¿las guardaste en una lista común?"
    shapes = [tuple(lin.weight.shape) for lin in linears]
    assert shapes == [(32, 64), (16, 32), (10, 16)], f"las capas deberían tener pesos (32, 64), (16, 32), (10, 16); tienen {shapes}"
    assert len(list(m.parameters())) == 6, "parameters() debería dar 6 tensores: un peso y un sesgo por capa"


def test_mlp_forward():
    torch.manual_seed(2)
    for sizes in ([64, 32, 16, 10], [5, 3], [4, 8, 8, 8, 2]):
        m = MLP(sizes)
        x = torch.randn(7, sizes[0])
        out = m(x)
        assert any(isinstance(mod, nn.Linear) for mod in m.modules()),             "PyTorch no ve ninguna nn.Linear: ¿las guardaste en una lista común?"
        assert out.shape == (7, sizes[-1]), f"con sizes={sizes}, la salida debería ser (7, {sizes[-1]}), es {tuple(out.shape)}"
        assert torch.allclose(out, run_by_hand(m, x), atol=1e-6), \
            f"con sizes={sizes}, la ida no coincide: ReLU después de cada capa menos la última"


check("MLP: las capas", test_mlp_structure)
check("MLP: la ida", test_mlp_forward)

## 4. Pérdida, optimizador y el bucle de cinco líneas

Faltan dos piezas para entrenar.

**La pérdida.** `nn.CrossEntropyLoss()` es la entropía cruzada de la semana 4. Recibe los **puntajes crudos** que salen de la red (en inglés, *logits*) y las etiquetas enteras, y aplica *softmax* por dentro. No le pases probabilidades: ya las calcula ella.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

scores = net(X_train_t)                                  # puntajes crudos: (1347, 10)
P = torch.softmax(scores, dim=1)
ours = -torch.log(P[torch.arange(len(y_train_t)), y_train_t]).mean()

print(f"nn.CrossEntropyLoss:     {loss_fn(scores, y_train_t).item():.4f}")
print(f"la fórmula de la semana 4: {ours.item():.4f}")
print(f"adivinar al azar, ln 10: {np.log(10):.4f}")

**El optimizador.** `torch.optim.SGD(model.parameters(), lr=...)` recibe la lista de parámetros y tiene dos métodos:

- `optimizer.zero_grad()` borra el `.grad` de todos los parámetros.
- `optimizer.step()` da el paso del descenso de gradiente, $\theta \leftarrow \theta - \eta\, \nabla L$, a todos a la vez.

Es el bloque `with torch.no_grad()` de la parte 2, pero para miles de parámetros. Comprobémoslo con un paso sobre los primeros sesgos de la capa de salida. (Un tensor que lleva gradiente no pasa directo a NumPy: `.detach()` lo desprende del grafo primero.)

In [ ]:
optimizer = torch.optim.SGD(net.parameters(), lr=0.5)
before = net.output.bias.detach().clone()

loss = loss_fn(net(X_train_t), y_train_t)
optimizer.zero_grad()
loss.backward()
optimizer.step()

print("b antes:           ", np.round(before[:4].numpy(), 4))
print("b − 0.5 · gradiente:", np.round((before - 0.5 * net.output.bias.grad)[:4].numpy(), 4))
print("b después de step: ", np.round(net.output.bias.detach()[:4].numpy(), 4))

### El bucle de entrenamiento

Con eso, un paso de entrenamiento son **cinco líneas**, siempre las mismas y siempre en este orden. Las vas a escribir todo el año, para cualquier red:

```python
scores = model(X)               # 1. ida
loss = loss_fn(scores, y)       # 2. cuánto nos equivocamos
optimizer.zero_grad()           # 3. borrar los gradientes del paso anterior
loss.backward()                 # 4. vuelta: backpropagation
optimizer.step()                # 5. un paso cuesta abajo
```

Entrenemos la red de los dígitos con la misma receta de la semana 7: 500 pasos con $\eta = 0.5$, mirando todos los ejemplos en cada paso.

In [ ]:
torch.manual_seed(SEED)
model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

full_losses = []
start = time.perf_counter()
for step in range(500):
    scores = model(X_train_t)                 # 1. ida
    loss = loss_fn(scores, y_train_t)         # 2. cuánto nos equivocamos
    optimizer.zero_grad()                     # 3. borrar los gradientes del paso anterior
    loss.backward()                           # 4. vuelta
    optimizer.step()                          # 5. un paso cuesta abajo
    full_losses.append(loss.item())
t_full = time.perf_counter() - start

with torch.no_grad():
    acc_full = (model(X_test_t).argmax(dim=1) == y_test_t).float().mean().item()
baseline = np.mean(y_test == np.bincount(y_train).argmax())

print(f"línea base (la clase más común):     accuracy de prueba {baseline:.3f}")
print(f"semana 7, nuestra vuelta en NumPy:   accuracy de prueba 0.969")
print(f"PyTorch, 500 pasos en {t_full:.1f} s:        accuracy de prueba {acc_full:.3f}")

fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot(full_losses, color="tab:blue")
ax.axhline(np.log(10), color="gray", linestyle="--", label="adivinar: ln 10")
ax.set(xlabel="paso", ylabel="entropía cruzada", title="Dígitos con PyTorch: la pérdida, paso a paso", ylim=(0, 2.6))
ax.legend(fontsize=8)
plt.show()

El mismo nivel que la semana 7 (no los mismos números, porque los pesos empiezan en otros valores al azar), y sin escribir la vuelta. Fíjate en el `torch.no_grad()` al medir la *accuracy*: para predecir no hace falta la vuelta, así que le pedimos a PyTorch que no anote nada.

## 5. `Dataset` y `DataLoader`: aprender de a bocados

Hasta ahora, cada paso miraba los 1347 dígitos. Con MNIST (parte 6) serían 60 000 imágenes por paso, y con los conjuntos de datos de la competencia, millones: no caben, y cada paso sería lentísimo.

La salida es la de cualquiera que estudia con fichas: no repasas el mazo entero antes de corregir una respuesta. Barajas, tomas un puñado de 32 fichas, aprendes de ellas, tomas el siguiente puñado. Cada puñado es un **lote** (*mini-batch*), y cuando terminaste el mazo completaste una **época** (*epoch*). El gradiente de un lote es una estimación ruidosa del gradiente de todos los datos, pero es muchísimo más barato, y en una época das muchos pasos en lugar de uno.

PyTorch reparte el trabajo en dos piezas:

- Un **`Dataset`** sabe cuántos ejemplos hay y te da el ejemplo número `i`.
- Un **`DataLoader`** baraja el `Dataset` y lo entrega en lotes.

In [ ]:
train_set = TensorDataset(X_train_t, y_train_t)
x0, y0 = train_set[0]
print("ejemplos:", len(train_set), "| el primero:", tuple(x0.shape), "y su etiqueta", y0.item())

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(SEED))
X_batch, y_batch = next(iter(train_loader))
print("lotes por época:", len(train_loader))
print("un lote:", tuple(X_batch.shape), tuple(y_batch.shape))
print("sus primeras etiquetas:", y_batch[:12].tolist())

`TensorDataset` sirve cuando los datos ya caben en un tensor. Cuando no (miles de imágenes en disco, textos), escribes tu propio `Dataset`: una clase con `__len__` y `__getitem__`, el mismo patrón de clases de la semana 1. El `DataLoader` no sabe ni le importa de dónde salen los ejemplos: pide los que necesita y los apila en un lote.

In [ ]:
class DigitsDataset(Dataset):
    """Los dígitos de sklearn, de a uno: una imagen 8 × 8 aplanada (float32) y su etiqueta."""

    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        x = torch.tensor(self.images[i], dtype=torch.float32)   # aquí podrías leer un archivo del disco
        return x, int(self.labels[i])


my_set = DigitsDataset(X_train, y_train)
x0, y0 = my_set[0]
print("ejemplos:", len(my_set), "| el primero:", tuple(x0.shape), x0.dtype, "y su etiqueta", y0)

X_batch, y_batch = next(iter(DataLoader(my_set, batch_size=32, shuffle=True)))
print("el DataLoader los apila:", tuple(X_batch.shape), tuple(y_batch.shape), y_batch.dtype)

Ahora las cinco líneas van dentro de un recorrido por los lotes. Las guardamos en dos funciones que vamos a usar el resto de la clase, y el resto del año:

- `train_epoch` hace una época de entrenamiento y devuelve la pérdida promedio de sus lotes.
- `accuracy` mide la fracción de aciertos sobre un `DataLoader`, dentro de `torch.no_grad()`.

`model.train()` y `model.eval()` avisan al modelo si está entrenando o evaluando. En estas redes no cambian nada, pero sí con *dropout* y *batch norm* (semanas 9 y 10): acostúmbrate a ponerlos. Las líneas con `device` sirven para la GPU (parte 7); mientras todo vive en la CPU, no hacen nada.

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    """Una época: una pasada por todos los lotes. Devuelve la pérdida promedio de los lotes."""
    model.train()
    device = next(model.parameters()).device                        # dónde vive el modelo (parte 7)
    total = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)   # el lote va a donde está el modelo
        scores = model(X_batch)                   # 1. ida
        loss = loss_fn(scores, y_batch)           # 2. cuánto nos equivocamos
        optimizer.zero_grad()                     # 3. borrar los gradientes del paso anterior
        loss.backward()                           # 4. vuelta
        optimizer.step()                          # 5. un paso cuesta abajo
        total += loss.item()
    return total / len(loader)


def accuracy(model, loader):
    """Fracción de aciertos sobre todos los ejemplos de loader, sin anotar nada para la vuelta."""
    model.eval()
    device = next(model.parameters()).device
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            predicted = model(X_batch.to(device)).argmax(dim=1).cpu()
            correct += (predicted == y_batch).sum().item()
    return correct / len(loader.dataset)

Entrenemos la misma red con lotes de 32, durante 20 épocas, y comparemos con el entrenamiento de la parte 4, que miraba todos los datos en cada paso.

In [ ]:
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=500)

torch.manual_seed(SEED)
model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

mini_losses = []
start = time.perf_counter()
for epoch in range(20):
    mini_losses.append(train_epoch(model, train_loader, loss_fn, optimizer))
t_mini = time.perf_counter() - start

print(f"todos los datos por paso: 500 épocas,  500 pasos en {t_full:.1f} s, accuracy de prueba {acc_full:.3f}")
print(f"lotes de 32:               20 épocas, {20 * len(train_loader)} pasos en {t_mini:.1f} s, accuracy de prueba {accuracy(model, test_loader):.3f}")

fig, ax = plt.subplots(figsize=(6, 3.6))
epochs = np.arange(1, 21)
ax.plot(epochs, full_losses[:20], "o-", ms=4, color="tab:gray", label="todos los datos por paso")
ax.plot(epochs, mini_losses, "o-", ms=4, color="tab:blue", label="lotes de 32")
ax.set(xlabel="época (pasadas por los datos)", ylabel="entropía cruzada", title="Las primeras 20 épocas", ylim=(0, 2.6))
ax.set_xticks([1, 5, 10, 15, 20])
ax.legend(fontsize=8)
plt.show()

Con la misma cantidad de datos vistos, los lotes aprenden muchísimo más: en una época dan 43 pasos en vez de uno. Con 20 pasadas por los datos llegan casi al mismo lugar que el lote completo con 500. Por eso, en la práctica, **siempre** se entrena con lotes. El tamaño del lote y la tasa de aprendizaje van juntos, y elegirlos bien es el tema de la semana 10.

### Tu turno 4: ¿por qué no aprende?

Escribir mal el bucle de entrenamiento casi nunca produce un error. La red entrena igual, solo que no aprende, o aprende mal, y nadie avisa.

`buggy_train_epoch` tiene **dos** errores. Uno hace que la red no aprenda nada; el otro, que aprenda mucho más lento. Ejecuta la celda y mira la pérdida. Después:

1. Encuentra los dos errores. Pistas: compara con las cinco líneas de la parte 4, y vuelve a leer qué recibe `nn.CrossEntropyLoss`.
2. Escribe `fixed_train_epoch`, una copia de `buggy_train_epoch` con los dos errores corregidos.
3. Pregunta para pensar: con el primer error, ¿qué hace exactamente `optimizer.step()`?

In [ ]:
def buggy_train_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total = 0.0
    for X_batch, y_batch in loader:
        scores = model(X_batch)
        probs = torch.softmax(scores, dim=1)
        loss = loss_fn(probs, y_batch)
        loss.backward()
        optimizer.zero_grad()
        optimizer.step()
        total += loss.item()
    return total / len(loader)


torch.manual_seed(SEED)
bug_model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
bug_optimizer = torch.optim.SGD(bug_model.parameters(), lr=0.2)
for epoch in range(5):
    print(f"época {epoch + 1}: pérdida {buggy_train_epoch(bug_model, train_loader, loss_fn, bug_optimizer):.3f}")
print(f"accuracy de prueba: {accuracy(bug_model, test_loader):.3f}")

In [ ]:
def fixed_train_epoch(model, loader, loss_fn, optimizer):
    # TODO
    raise NotImplementedError

In [ ]:
def test_one_step():
    torch.manual_seed(1)
    mine = nn.Sequential(nn.Linear(64, 8), nn.ReLU(), nn.Linear(8, 10))
    ref = copy.deepcopy(mine)
    one_batch = [(X_train_t[:32], y_train_t[:32])]           # un "loader" de un solo lote
    fixed_train_epoch(mine, one_batch, loss_fn, torch.optim.SGD(mine.parameters(), lr=0.2))
    ref_opt = torch.optim.SGD(ref.parameters(), lr=0.2)
    ref_opt.zero_grad()
    loss_fn(ref(X_train_t[:32]), y_train_t[:32]).backward()
    ref_opt.step()
    for (name, p), q in zip(mine.named_parameters(), ref.parameters()):
        assert torch.allclose(p, q, atol=1e-6), f"después de un paso, {name} no quedó donde debía"


def test_learns():
    torch.manual_seed(SEED)
    m = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
    opt = torch.optim.SGD(m.parameters(), lr=0.2)
    loader = DataLoader(train_set, batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(SEED))
    for _ in range(20):
        fixed_train_epoch(m, loader, loss_fn, opt)
    acc = accuracy(m, test_loader)
    assert acc > 0.93, f"después de 20 épocas, accuracy de prueba {acc:.3f}; debería pasar de 0.93"


check("fixed_train_epoch: un paso", test_one_step)
check("fixed_train_epoch: aprende", test_learns)

## 6. Un problema de verdad: MNIST

Los dígitos de sklearn son imágenes de 8 × 8, y hay 1797. **MNIST** es el conjunto de datos clásico de dígitos escritos a mano: 70 000 imágenes de 28 × 28, 60 000 para entrenar y 10 000 para probar. Viene en `torchvision`, la librería de imágenes de PyTorch, y se descarga solo la primera vez (unos 12 MB).

Cada imagen llega como una matriz de 28 × 28 enteros entre 0 y 255. Igual que con los dígitos, la aplanamos en 784 números y la dividimos entre 255 para que queden entre 0 y 1. Las 60 000 caben en memoria, así que las guardamos en un solo tensor y usamos `TensorDataset`.

In [ ]:
mnist_train = datasets.MNIST(root="data", train=True, download=True)
mnist_test = datasets.MNIST(root="data", train=False, download=True)
print("imágenes:", tuple(mnist_train.data.shape), mnist_train.data.dtype, "| etiquetas:", mnist_train.targets[:10].tolist())

Xm_train = mnist_train.data.reshape(-1, 784).float() / 255      # cada imagen 28 × 28 → 784 números entre 0 y 1
Xm_test = mnist_test.data.reshape(-1, 784).float() / 255
ym_train, ym_test = mnist_train.targets, mnist_test.targets

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for ax, img, label in zip(axes, mnist_train.data[:10], ym_train[:10]):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(label.item())
    ax.axis("off")
plt.show()

Tres modelos, con las mismas funciones de la parte 5 y 3 épocas cada uno:

- La **línea base**: siempre la clase más común.
- Un **modelo lineal**, `nn.Linear(784, 10)`: diez puntajes, uno por dígito, sin capa oculta. Con la entropía cruzada es la regresión logística de la semana 5, entrenada con lotes.
- Un **MLP**, $784 \to 128 \to 10$.

In [ ]:
mnist_loader = DataLoader(TensorDataset(Xm_train, ym_train), batch_size=64, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
mnist_test_loader = DataLoader(TensorDataset(Xm_test, ym_test), batch_size=1000)


def fit(model, epochs, lr=0.1):
    """Entrena model sobre MNIST con SGD y muestra la pérdida y la accuracy de prueba de cada época."""
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    start = time.perf_counter()
    for epoch in range(epochs):
        loss = train_epoch(model, mnist_loader, loss_fn, optimizer)
        print(f"  época {epoch + 1}: pérdida {loss:.3f}, accuracy de prueba {accuracy(model, mnist_test_loader):.3f}")
    print(f"  ({time.perf_counter() - start:.1f} s)")


majority = ym_train.bincount().argmax()
print(f"línea base (siempre {majority.item()}): accuracy de prueba {(ym_test == majority).float().mean().item():.3f}")

torch.manual_seed(SEED)
linear = nn.Linear(784, 10)
print("modelo lineal 784 → 10:")
fit(linear, epochs=3)

torch.manual_seed(SEED)
mlp = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
print("MLP 784 → 128 → 10:")
fit(mlp, epochs=3)

La capa oculta baja el error de un 8 % a un 4.4 %: casi a la mitad. Y todavía no hicimos nada para entrenar bien (eso es la semana 10). Veamos en qué se equivoca el MLP:

In [ ]:
with torch.no_grad():
    predicted = mlp(Xm_test).argmax(dim=1)
wrong = torch.nonzero(predicted != ym_test).flatten()
print(f"errores: {len(wrong)} de {len(ym_test)}")

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, i in zip(axes, wrong[:8]):
    ax.imshow(mnist_test.data[i], cmap="gray_r")
    ax.set_title(f"dijo {predicted[i].item()}, era {ym_test[i].item()}", fontsize=9)
    ax.axis("off")
plt.show()

Algunos errores son disculpables, y otros no tanto. Un MLP mira los 784 píxeles como una lista sin vecinos: no sabe que un píxel está al lado de otro. Las redes convolucionales (semana 13) sí lo saben, y en MNIST pasan del 99 %.

## 7. CPU vs GPU

Una **CPU** es como un puñado de chefs expertos: pocos, pero cada uno rapidísimo y capaz de cualquier receta. Una **GPU** es un estadio lleno de miles de ayudantes que solo saben hacer sumas y multiplicaciones simples, pero todos a la vez. Para un sándwich, el chef gana: mientras al estadio le explicas qué hacer, el chef ya terminó. Para diez mil sándwiches iguales, gana el estadio.

Multiplicar matrices grandes es el caso de los diez mil sándwiches: millones de multiplicaciones independientes. Y entrenar una red es, casi todo, multiplicar matrices.

En PyTorch, cada tensor vive en un **dispositivo** (*device*): `"cpu"` o `"cuda"` (la GPU). Esta celda elige la GPU si hay una.

**En Colab**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4. El entorno se reinicia, así que después hay que volver a ejecutar todo desde arriba. Si no la activas, esta parte funciona igual, solo con la CPU.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cpu = torch.device("cpu")
print("dispositivo:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No hay GPU: las comparaciones de abajo solo muestran la CPU.")

Midamos cuánto tarda una multiplicación de dos matrices $n \times n$ en cada dispositivo. Un detalle al medir: la GPU trabaja en paralelo con Python. Python le encarga la cuenta y sigue de largo, así que para cronometrarla hay que esperarla con `torch.cuda.synchronize()`. Y la primera cuenta en la GPU es más lenta, porque la prepara: la hacemos una vez antes de medir.

In [ ]:
def time_matmul(n, device, reps=5):
    """Segundos que tarda A @ B con dos matrices n × n en device (promedio de reps)."""
    A = torch.randn(n, n, device=device)
    B = torch.randn(n, n, device=device)
    A @ B                                     # calentar
    if device.type == "cuda":
        torch.cuda.synchronize()              # esperar a que la GPU termine
    start = time.perf_counter()
    for _ in range(reps):
        A @ B
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / reps


sizes = [64, 256, 1024, 2048]
devices = [cpu] + ([device] if device.type == "cuda" else [])
times = {d.type: [time_matmul(n, d) for n in sizes] for d in devices}

for i, n in enumerate(sizes):
    line = f"{n:>4} × {n:<4}  CPU {1000 * times['cpu'][i]:8.2f} ms"
    if "cuda" in times:
        line += f"   GPU {1000 * times['cuda'][i]:6.2f} ms   CPU / GPU = {times['cpu'][i] / times['cuda'][i]:5.1f}"
    print(line)

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.loglog(sizes, [1000 * s for s in times["cpu"]], "o-", color="tab:blue", label="CPU")
if "cuda" in times:
    ax.loglog(sizes, [1000 * s for s in times["cuda"]], "o-", color="tab:orange", label="GPU")
ax.set_xticks(sizes, [str(n) for n in sizes])
ax.set(xlabel="n (matrices n × n)", ylabel="milisegundos", title="Multiplicar dos matrices")
ax.legend(fontsize=8)
plt.show()

Con matrices chicas, la GPU no gana nada, o hasta pierde: encargarle la cuenta cuesta más que hacerla. Con matrices grandes, gana por mucho, y la ventaja crece con el tamaño.

### Mover el modelo y los datos

Para usar la GPU, el modelo y los datos tienen que estar **en el mismo dispositivo**: `model.to(device)` mueve los parámetros y `X.to(device)` mueve los datos. Si se te olvida uno, PyTorch no adivina: da un error. Es el segundo error que vas a ver más seguido, después del de los tipos.

In [ ]:
model_dev = nn.Linear(64, 10).to(device)
try:
    model_dev(X_test_t)                        # el modelo en device, los datos en la CPU
    print("Sin error: aquí el modelo y los datos están los dos en la CPU.")
except RuntimeError as e:
    print("Error:", e)

print("con X_test_t.to(device):", tuple(model_dev(X_test_t.to(device)).shape))

Por eso `train_epoch` y `accuracy` miran dónde vive el modelo y mandan cada lote allá: con ellas, entrenar en la GPU es solo `model.to(device)`.

¿Vale la pena para cualquier red? Comparemos una época de MNIST con dos redes: el MLP de la parte 6, con lotes de 64, y una red ancha, $784 \to 1024 \to 1024 \to 10$, con lotes de 256.

In [ ]:
def time_epoch(make_model, batch_size, device):
    """Segundos que tarda una época de MNIST con un modelo nuevo en device."""
    torch.manual_seed(SEED)
    model = make_model().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    loader = DataLoader(TensorDataset(Xm_train, ym_train), batch_size=batch_size, shuffle=True)
    start = time.perf_counter()
    train_epoch(model, loader, loss_fn, optimizer)
    return time.perf_counter() - start


def small_net():
    return nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))


def wide_net():
    return nn.Sequential(nn.Linear(784, 1024), nn.ReLU(), nn.Linear(1024, 1024), nn.ReLU(), nn.Linear(1024, 10))


for name, make_model, batch_size in [("chica, lotes de 64", small_net, 64), ("ancha, lotes de 256", wide_net, 256)]:
    n_params = sum(p.numel() for p in make_model().parameters())
    line = f"red {name:<20} ({n_params:>9,} parámetros):  CPU {time_epoch(make_model, batch_size, cpu):5.1f} s"
    if device.type == "cuda":
        line += f"   GPU {time_epoch(make_model, batch_size, device):5.1f} s"
    print(line)

Si tienes la GPU activa, mira las dos filas. Con la red chica, la GPU gana poco o incluso pierde: cada lote es tan poca cuenta que se va el tiempo en encargarla y en mover el lote. Con la red ancha, la GPU gana con claridad.

La regla: **la GPU conviene cuando hay mucha cuenta por paso**: redes grandes, lotes grandes, imágenes. Desde la semana 13, con redes convolucionales, va a ser indispensable; en el Sprint 1 (semana 11) vas a tener 15 minutos de GPU, y saber usarlos bien es parte del problema.

## 8. Cierre

**Hoy vimos:**

- Un tensor es un arreglo de NumPy con memoria (`autograd`) y pasaporte (`.to(device)`). Entradas en `float32`, etiquetas en `int64`; `axis` se dice `dim`.
- `requires_grad=True` hace que PyTorch anote la cuenta; `loss.backward()` recorre ese grafo hacia atrás y deja cada gradiente en `.grad`. Es la vuelta de la semana 7 y el motor de P1.
- Los gradientes se **acumulan**: hay que borrarlos en cada paso. Y el paso se da dentro de `torch.no_grad()`.
- `nn.Linear`, `nn.Sequential` y las clases que heredan de `nn.Module` arman la red; sus parámetros se registran solos (nunca en una lista común de Python).
- `nn.CrossEntropyLoss` recibe puntajes crudos, no probabilidades. `optimizer.zero_grad()` y `optimizer.step()` hacen el descenso de gradiente sobre todos los parámetros.
- El bucle de cinco líneas: ida, pérdida, `zero_grad`, `backward`, `step`. Dentro de un recorrido por los lotes de un `DataLoader`, eso es una época.
- La GPU conviene cuando hay mucha cuenta por paso. El modelo y los datos deben estar en el mismo dispositivo.

**Un poco de historia.** PyTorch nació en 2016 en el laboratorio de inteligencia artificial de Facebook (hoy Meta), heredero de Torch, una librería anterior escrita en el lenguaje Lua. Su idea central fue que el grafo se arma **mientras** se ejecuta la cuenta, como hoy en la parte 2, en lugar de declararlo antes. Hoy la mayoría de los artículos de investigación en aprendizaje profundo usan PyTorch.

**Para la semana 9:**

1. **Entrega al inicio de la clase**: **P2: MLP sobre Fashion-MNIST + tabla de ablación**. Fashion-MNIST tiene la misma forma que MNIST (28 × 28, 10 clases), pero con ropa en vez de dígitos, y es bastante más difícil. Vas a entrenar un MLP con el bucle de hoy y cambiar **una sola cosa a la vez** (la *ablación*) para medir cuánto aporta cada una. El enunciado se publica en la carpeta `Nivel_Intermedio/tareas/` de este repositorio; se guarda en tu repositorio de entregas como `P2/`.
2. **Cuota semanal** en [la plataforma](https://ioai.artix.tech/).
3. **La semana 9 es un seminario** [C]: presentan tres grupos, sobre funciones de pérdida (MSE vs MAE vs entropía cruzada), inicialización de pesos y *batch norm*.
4. **Para leer** (en inglés, opcional): [Learn the Basics](https://pytorch.org/tutorials/beginner/basics/intro.html), el tutorial oficial de PyTorch. Cubre lo mismo que hoy, con Fashion-MNIST.